# Cardiovascular Disease Prediction

## 1. Introduction

This Jupyter notebook aims to build a machine learning model to predict the 10-year risk of developing Coronary Heart Disease (CHD) based on various patient attributes. The dataset contains a mix of demographic, behavioral, and medical information.

Coronary Heart Disease (CHD) is a major health concern globally. Early prediction can help in timely intervention and management, potentially saving lives and improving quality of life. This project will involve data exploration, preprocessing, model training, evaluation, and interpretation to identify key risk factors and build a robust predictive model.

**Dataset Features:**
*   `male`: (Binary) 0 = Female, 1 = Male
*   `age`: Age of the patient (Continuous)
*   `education`: Educational level (Categorical: 1=some high school, 2=high school/GED, 3=some college/vocational school, 4=college)
*   `currentSmoker`: (Binary) 0 = non-smoker, 1 = smoker
*   `cigsPerDay`: Number of cigarettes smoked per day (Continuous)
*   `BPMeds`: Whether the patient is on blood pressure medication (Binary)
*   `prevalentStroke`: History of stroke (Binary)
*   `prevalentHyp`: History of hypertension (Binary)
*   `diabetes`: History of diabetes (Binary)
*   `totChol`: Total cholesterol level (Continuous)
*   `sysBP`: Systolic blood pressure (Continuous)
*   `diaBP`: Diastolic blood pressure (Continuous)
*   `BMI`: Body Mass Index (Continuous)
*   `heartRate`: Heart rate (Continuous)
*   `glucose`: Glucose level (Continuous)
*   `TenYearCHD`: (Target Variable) 10-year risk of Coronary Heart Disease (Binary: 0 = No CHD, 1 = CHD)

Our primary goal is to predict the `TenYearCHD` variable, which makes this a binary classification problem.


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix, classification_report
from imblearn.over_sampling import SMOTE
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set display options for pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)


## 2. Data Loading

We will load the dataset from the specified CSV file. For this notebook, we assume the dataset is named 'framingham.csv'. If your file has a different name, please update the `file_path` variable.


In [ ]:
# Define the file path
file_path = 'framingham.csv' # Please ensure this file is in the same directory or provide the full path

# Load the dataset
try:
    df = pd.read_csv(file_path)
    print(f"Dataset loaded successfully from '{file_path}'. Shape: {df.shape}")
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please ensure the CSV file is in the correct directory.")
    # Exit or create a dummy dataframe for demonstration purposes
    # For now, we will create a dummy dataframe based on the sample data if the file is not found
    sample_data = [{'male': 1, 'age': 39, 'education': 4.0, 'currentSmoker': 0, 'cigsPerDay': 0.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 0, 'diabetes': 0, 'totChol': 195.0, 'sysBP': 106.0, 'diaBP': 70.0, 'BMI': 26.97, 'heartRate': 80.0, 'glucose': 77.0, 'TenYearCHD': 0}, {'male': 0, 'age': 46, 'education': 2.0, 'currentSmoker': 0, 'cigsPerDay': 0.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 0, 'diabetes': 0, 'totChol': 250.0, 'sysBP': 121.0, 'diaBP': 81.0, 'BMI': 28.73, 'heartRate': 95.0, 'glucose': 76.0, 'TenYearCHD': 0}, {'male': 1, 'age': 48, 'education': 1.0, 'currentSmoker': 1, 'cigsPerDay': 20.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 0, 'diabetes': 0, 'totChol': 245.0, 'sysBP': 127.5, 'diaBP': 80.0, 'BMI': 25.34, 'heartRate': 75.0, 'glucose': 70.0, 'TenYearCHD': 0}, {'male': 0, 'age': 61, 'education': 3.0, 'currentSmoker': 1, 'cigsPerDay': 30.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 1, 'diabetes': 0, 'totChol': 225.0, 'sysBP': 150.0, 'diaBP': 95.0, 'BMI': 28.58, 'heartRate': 65.0, 'glucose': 103.0, 'TenYearCHD': 1}, {'male': 0, 'age': 46, 'education': 3.0, 'currentSmoker': 1, 'cigsPerDay': 23.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 0, 'diabetes': 0, 'totChol': 285.0, 'sysBP': 130.0, 'diaBP': 84.0, 'BMI': 23.1, 'heartRate': 85.0, 'glucose': 85.0, 'TenYearCHD': 0}, {'male': 0, 'age': 43, 'education': 2.0, 'currentSmoker': 0, 'cigsPerDay': 0.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 1, 'diabetes': 0, 'totChol': 228.0, 'sysBP': 180.0, 'diaBP': 110.0, 'BMI': 30.3, 'heartRate': 77.0, 'glucose': 99.0, 'TenYearCHD': 0}, {'male': 0, 'age': 63, 'education': 1.0, 'currentSmoker': 0, 'cigsPerDay': 0.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 0, 'diabetes': 0, 'totChol': 205.0, 'sysBP': 138.0, 'diaBP': 71.0, 'BMI': 33.11, 'heartRate': 60.0, 'glucose': 85.0, 'TenYearCHD': 1}, {'male': 0, 'age': 45, 'education': 2.0, 'currentSmoker': 1, 'cigsPerDay': 20.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 0, 'diabetes': 0, 'totChol': 313.0, 'sysBP': 100.0, 'diaBP': 71.0, 'BMI': 21.68, 'heartRate': 79.0, 'glucose': 78.0, 'TenYearCHD': 0}, {'male': 1, 'age': 52, 'education': 1.0, 'currentSmoker': 0, 'cigsPerDay': 0.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 1, 'diabetes': 0, 'totChol': 260.0, 'sysBP': 141.5, 'diaBP': 89.0, 'BMI': 26.36, 'heartRate': 76.0, 'glucose': 79.0, 'TenYearCHD': 0}, {'male': 1, 'age': 43, 'education': 1.0, 'currentSmoker': 1, 'cigsPerDay': 30.0, 'BPMeds': 0.0, 'prevalentStroke': 0, 'prevalentHyp': 1, 'diabetes': 0, 'totChol': 225.0, 'sysBP': 162.0, 'diaBP': 107.0, 'BMI': 23.61, 'heartRate': 93.0, 'glucose': 88.0, 'TenYearCHD': 0}]
    df = pd.DataFrame(sample_data)
    print("Loaded sample data as a fallback. Please provide the actual 'framingham.csv' for full analysis.")


# Display the first 5 rows of the dataset
print("\nFirst 5 rows of the dataset:")
print(df.head())


## 3. Exploratory Data Analysis (EDA)

EDA is a critical step to understand the dataset's structure, identify patterns, and detect anomalies.


In [ ]:
# Display basic information about the dataset
print("\nDataset Info:")
df.info()

# Display descriptive statistics for numerical columns
print("\nDescriptive Statistics:")
print(df.describe().T)

# Check for missing values
print("\nMissing Values Count:")
print(df.isnull().sum())

print("\nMissing Values Percentage:")
print((df.isnull().sum() / len(df)) * 100)

# Check distribution of the target variable
print("\nTarget Variable (TenYearCHD) Distribution:")
print(df['TenYearCHD'].value_counts())
print("\nTarget Variable (TenYearCHD) Percentage Distribution:")
print(df['TenYearCHD'].value_counts(normalize=True) * 100)


### EDA Observations:
*   The dataset contains 4238 entries and 16 columns.
*   Several columns like `education`, `cigsPerDay`, `BPMeds`, `totChol`, `BMI`, `heartRate`, and `glucose` have missing values. `glucose` has the highest percentage of missing values.
*   Data types are mostly `int64` and `float64`, which are appropriate for numerical and binary features. `education` is `float64` but represents categorical levels; it might be treated as a categorical feature or ordinal.
*   The target variable `TenYearCHD` shows a significant imbalance: a large majority (around 85%) do not develop CHD, while a smaller portion (around 15%) do. This imbalance will need to be addressed during model training to prevent biased predictions.


## 4. Preprocessing

Based on our EDA, we need to perform the following preprocessing steps:
1.  **Handle Missing Values**: Impute missing values for numerical features.
2.  **Feature Engineering**: Potentially create new features from existing ones.
3.  **Handle Categorical Features**: Convert `education` into an appropriate format. Other binary features are already encoded.
4.  **Feature Scaling**: Scale numerical features to ensure no single feature dominates the model training.
5.  **Handle Class Imbalance**: Address the imbalance in the target variable `TenYearCHD`.


In [ ]:
# Make a copy of the dataframe for preprocessing
df_processed = df.copy()

# 1. Handle Missing Values
# Impute numerical missing values using the median, as it's more robust to outliers than the mean.
numerical_cols_with_missing = ['cigsPerDay', 'totChol', 'BMI', 'heartRate', 'glucose', 'BPMeds']
# 'education' is categorical, we will impute with mode
categorical_cols_with_missing = ['education']

for col in numerical_cols_with_missing:
    median_val = df_processed[col].median()
    df_processed[col].fillna(median_val, inplace=True)
    print(f"Missing values in '{col}' imputed with median: {median_val}")

for col in categorical_cols_with_missing:
    mode_val = df_processed[col].mode()[0] # .mode() can return multiple if frequencies are tied
    df_processed[col].fillna(mode_val, inplace=True)
    print(f"Missing values in '{col}' imputed with mode: {mode_val}")


# Verify no more missing values
print("\nMissing Values after imputation:")
print(df_processed.isnull().sum())

# 2. Feature Engineering (Example)
# A simple feature engineering step: combine age and blood pressure to create a "risk_score" (hypothetical)
# Or create interaction terms, e.g., 'age_smoker' = age * currentSmoker
df_processed['age_cigs_per_day'] = df_processed['age'] * df_processed['cigsPerDay']
df_processed['BMI_BP'] = df_processed['BMI'] * df_processed['sysBP'] / df_processed['diaBP']
print("\nAdded 'age_cigs_per_day' and 'BMI_BP' as engineered features.")

# 3. Handle Categorical Features
# 'education' is ordinal, so we can treat it as such or one-hot encode.
# Given it's 1-4, we'll keep it as is, treating it as numerical ordinal, but could one-hot encode if needed.
# For now, we'll confirm its type.
print(f"\n'education' column type: {df_processed['education'].dtype}")
# If education was 'object', we would use one-hot encoding:
# df_processed = pd.get_dummies(df_processed, columns=['education'], prefix='education', drop_first=True)

# 4. Feature Scaling (will be applied after train-test split to prevent data leakage)
# Identify numerical columns for scaling (excluding binary and 'education' if treated as ordinal)
numerical_features = ['age', 'cigsPerDay', 'totChol', 'sysBP', 'diaBP', 'BMI', 'heartRate', 'glucose', 'age_cigs_per_day', 'BMI_BP']

# Separate features (X) and target (y) before scaling
X = df_processed.drop('TenYearCHD', axis=1)
y = df_processed['TenYearCHD']

print("\nFeatures (X) and Target (y) separated.")
print(f"Shape of X: {X.shape}, Shape of y: {y.shape}")


## 5. Visual Representation of EDA

We will use Plotly to create interactive and informative visualizations to better understand the data distribution and relationships.


In [ ]:
# Distribution of Age
fig = px.histogram(df_processed, x='age', nbins=20, title='Distribution of Age',
                   labels={'age': 'Age (Years)'}, color_discrete_sequence=['#4287f5'])
fig.update_layout(bargap=0.1)
fig.show()

# Distribution of TenYearCHD
fig = px.pie(df_processed, names='TenYearCHD', title='Distribution of TenYearCHD (Target Variable)',
             color_discrete_sequence=px.colors.sequential.RdBu)
fig.show()

# Distribution of key numerical features
fig = make_subplots(rows=3, cols=2, subplot_titles=('Total Cholesterol', 'Systolic BP', 'Diastolic BP', 'BMI', 'Heart Rate', 'Glucose'))

fig.add_trace(go.Histogram(x=df_processed['totChol'], name='TotChol', marker_color='#ff7f0e'), row=1, col=1)
fig.add_trace(go.Histogram(x=df_processed['sysBP'], name='SysBP', marker_color='#2ca02c'), row=1, col=2)
fig.add_trace(go.Histogram(x=df_processed['diaBP'], name='DiaBP', marker_color='#d62728'), row=2, col=1)
fig.add_trace(go.Histogram(x=df_processed['BMI'], name='BMI', marker_color='#9467bd'), row=2, col=2)
fig.add_trace(go.Histogram(x=df_processed['heartRate'], name='HeartRate', marker_color='#8c564b'), row=3, col=1)
fig.add_trace(go.Histogram(x=df_processed['glucose'], name='Glucose', marker_color='#e377c2'), row=3, col=2)

fig.update_layout(title_text="Distribution of Key Medical Features", height=800, showlegend=False)
fig.show()

# Binary features vs. TenYearCHD
binary_features = ['male', 'currentSmoker', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes']

for feature in binary_features:
    df_agg = df_processed.groupby(feature)['TenYearCHD'].value_counts(normalize=True).mul(100).rename('percent').reset_index()
    df_agg_chd = df_agg[df_agg['TenYearCHD'] == 1] # Focus on those with CHD

    fig = px.bar(df_agg_chd, x=feature, y='percent', color=feature,
                 title=f'Percentage of CHD for {feature}',
                 labels={feature: feature, 'percent': 'Percentage with CHD (%)'},
                 color_discrete_sequence=px.colors.qualitative.Pastel)
    fig.show()

# Age vs TenYearCHD
fig = px.box(df_processed, x='TenYearCHD', y='age', points='all',
             title='Age Distribution by TenYearCHD Status',
             labels={'TenYearCHD': '10-Year CHD Risk (0=No, 1=Yes)', 'age': 'Age'},
             color='TenYearCHD', color_discrete_sequence=['blue', 'red'])
fig.show()

# Cholesterol vs TenYearCHD
fig = px.box(df_processed, x='TenYearCHD', y='totChol', points='all',
             title='Total Cholesterol Distribution by TenYearCHD Status',
             labels={'TenYearCHD': '10-Year CHD Risk (0=No, 1=Yes)', 'totChol': 'Total Cholesterol'},
             color='TenYearCHD', color_discrete_sequence=['blue', 'red'])
fig.show()

# Systolic BP vs TenYearCHD
fig = px.box(df_processed, x='TenYearCHD', y='sysBP', points='all',
             title='Systolic Blood Pressure Distribution by TenYearCHD Status',
             labels={'TenYearCHD': '10-Year CHD Risk (0=No, 1=Yes)', 'sysBP': 'Systolic BP'},
             color='TenYearCHD', color_discrete_sequence=['blue', 'red'])
fig.show()

# Glucose vs TenYearCHD
fig = px.box(df_processed, x='TenYearCHD', y='glucose', points='all',
             title='Glucose Distribution by TenYearCHD Status',
             labels={'TenYearCHD': '10-Year CHD Risk (0=No, 1=Yes)', 'glucose': 'Glucose'},
             color='TenYearCHD', color_discrete_sequence=['blue', 'red'])
fig.show()


### Explanations of EDA Visualizations:

*   **Distribution of Age**: Shows a relatively normal distribution of ages, mostly concentrating between 40 and 60 years old in the dataset.
*   **Distribution of TenYearCHD**: Clearly highlights the class imbalance, with a small percentage of individuals developing CHD. This confirms the need for imbalance handling.
*   **Distribution of Key Medical Features**: Histograms for `totChol`, `sysBP`, `diaBP`, `BMI`, `heartRate`, and `glucose` show their respective value ranges and most frequent values. Some distributions, like `sysBP`, might have a slight skew towards higher values, which could be indicative of risk factors.
*   **Binary features vs. TenYearCHD**: These bar plots show the proportion of individuals with CHD for each category of the binary features.
    *   **Male**: Males seem to have a slightly higher percentage of CHD than females (if 'male' is 1 for male).
    *   **Current Smoker**: Current smokers tend to have a higher percentage of CHD, as expected.
    *   **BPMeds, Prevalent Stroke, Prevalent Hyp, Diabetes**: Individuals with these conditions (value=1) show significantly higher percentages of CHD, indicating strong risk factors.
*   **Age, Total Cholesterol, Systolic BP, Glucose Distribution by TenYearCHD Status**: Box plots reveal differences in the distributions of these continuous variables between the 'No CHD' (0) and 'CHD' (1) groups.
    *   **Age**: The median age for individuals with CHD is noticeably higher than for those without, suggesting age is a strong risk factor.
    *   **Total Cholesterol, Systolic BP, Glucose**: The box plots for these features show that individuals who develop CHD generally have higher median values and wider ranges in these metrics compared to those who don't. This suggests that elevated cholesterol, blood pressure, and glucose levels are important predictors of CHD.

These visualizations confirm intuitive medical knowledge and highlight the importance of age, smoking status, existing medical conditions (hypertension, diabetes, stroke), and biometric measurements (cholesterol, BP, glucose) in predicting CHD risk.


## 6. Visual Representation of Correlation and Covariance

Correlation measures the strength and direction of a linear relationship between two variables. Covariance measures how two variables change together. A positive covariance indicates that variables tend to move in the same direction, while a negative covariance indicates they tend to move in opposite directions. Correlation is a standardized version of covariance, making it easier to compare the strength of relationships across different pairs of variables.


In [ ]:
# Select numerical columns for correlation and covariance matrices
# Exclude binary features from continuous numerical correlation if they are better seen in group comparisons
# We include all processed features including engineered ones for now
numerical_and_binary_features = df_processed.drop('education', axis=1).columns.tolist() # Education is ordinal, can be included or excluded
numerical_and_binary_features.remove('TenYearCHD') # Exclude target for correlation *between features*

# Convert relevant columns to numeric if they are not already (e.g., if education was object type after imputation)
for col in numerical_and_binary_features:
    if df_processed[col].dtype == 'object':
        df_processed[col] = pd.to_numeric(df_processed[col], errors='coerce')


# Drop rows with any remaining NaNs if conversion introduced them (should not happen after imputation)
df_temp_corr = df_processed[numerical_and_binary_features + ['TenYearCHD']].dropna()

# Calculate Correlation Matrix
correlation_matrix = df_temp_corr.corr()

# Plotting Correlation Matrix
fig_corr = px.imshow(correlation_matrix,
                     text_auto=True,
                     aspect="auto",
                     color_continuous_scale=px.colors.sequential.RdBu_r,
                     title='Correlation Matrix of Features and Target')
fig_corr.update_layout(height=800, width=900)
fig_corr.show()

# Calculate Covariance Matrix
covariance_matrix = df_temp_corr.cov()

# Plotting Covariance Matrix
fig_cov = px.imshow(covariance_matrix,
                     text_auto=True,
                     aspect="auto",
                     color_continuous_scale=px.colors.sequential.Plasma,
                     title='Covariance Matrix of Features and Target')
fig_cov.update_layout(height=800, width=900)
fig_cov.show()


### Explanation of Correlation and Covariance Plots:

**Correlation Matrix:**
*   **Interpretation:** The heatmap displays the Pearson correlation coefficient between all pairs of variables, ranging from -1 (perfect negative correlation) to 1 (perfect positive correlation). Values close to 0 indicate a weak linear relationship.
*   **Observations:**
    *   **Target Variable (`TenYearCHD`) Correlations:**
        *   `age`, `sysBP`, `diaBP`, `glucose`, `totChol`, `BMI`, `heartRate` show positive correlations with `TenYearCHD`, indicating that higher values in these features are associated with a higher risk of CHD. `sysBP` and `age` appear to have relatively stronger positive correlations.
        *   `male`, `currentSmoker`, `BPMeds`, `prevalentHyp`, `diabetes` also show positive correlations, reinforcing their roles as risk factors. `prevalentHyp` and `diabetes` show relatively strong correlations.
        *   `education` shows a negative correlation with `TenYearCHD`, suggesting that higher education levels are associated with a lower risk of CHD (though this could be an indirect effect of socioeconomic factors).
    *   **Feature-to-Feature Correlations (Multicollinearity):**
        *   `sysBP` and `diaBP` are highly positively correlated, which is expected as they are both blood pressure measurements.
        *   `currentSmoker` and `cigsPerDay` are also highly correlated, indicating that if someone is a current smoker, they likely have a positive number of cigarettes per day.
        *   `age` shows moderate positive correlations with `sysBP`, `totChol`, and `glucose`, which is also medically plausible.
        *   The engineered features `age_cigs_per_day` and `BMI_BP` show high correlation with their parent features, which is normal.
*   **Implications:** High correlations with the target variable suggest important predictors. High correlations between independent features (multicollinearity) can sometimes affect certain models (e.g., Logistic Regression) but may not be an issue for tree-based models.

**Covariance Matrix:**
*   **Interpretation:** This heatmap shows the unstandardized relationship between variables. The values themselves are harder to interpret directly compared to correlation coefficients, as they depend on the scale of the variables. However, the sign (positive/negative) indicates the direction of the relationship.
*   **Observations:**
    *   The pattern of positive and negative relationships generally aligns with the correlation matrix.
    *   Variables with larger scales (e.g., `sysBP`, `totChol`) will tend to have larger covariance values.
*   **Implications:** While correlation is generally preferred for understanding relationship strength, covariance still indicates how variables vary together. Both matrices confirm the interdependencies and predictive potential of various features in the dataset.


## 7. Feature Selection Based on EDA

Based on the EDA and correlation analysis, we can select features that show strong relationships with the target variable and avoid highly redundant features if necessary.

**Selected Features and Rationale:**
*   **`age`**: Strong positive correlation with CHD, evident from box plots and correlation matrix. Older individuals are at higher risk.
*   **`male`**: Shows an association with CHD risk.
*   **`currentSmoker`**: Directly linked to cardiovascular health; strong correlation.
*   **`cigsPerDay`**: High correlation with `currentSmoker` but provides more granularity. We can keep both or choose one. Given the context, `cigsPerDay` is more quantitative.
*   **`BPMeds`**: Indicates existing medical intervention for blood pressure, a direct risk factor.
*   **`prevalentStroke`**: History of stroke is a major risk factor for future cardiovascular events.
*   **`prevalentHyp`**: History of hypertension is a strong indicator of CHD risk.
*   **`diabetes`**: A significant risk factor for cardiovascular diseases.
*   **`totChol`**: Higher cholesterol levels are a known risk factor.
*   **`sysBP`**: Higher systolic blood pressure is a strong indicator of CHD.
*   **`diaBP`**: Highly correlated with `sysBP`, but provides additional blood pressure information. We could potentially keep `sysBP` as the primary BP indicator or use both. For now, we keep both.
*   **`BMI`**: Obesity and higher BMI are associated with increased CHD risk.
*   **`heartRate`**: Elevated resting heart rate can be a risk factor.
*   **`glucose`**: High glucose levels (diabetes indicator) are linked to CHD.
*   **`education`**: While negatively correlated, it provides socioeconomic context that could influence lifestyle and health outcomes. We will keep it as an ordinal feature.
*   **Engineered features (`age_cigs_per_day`, `BMI_BP`)**: These combine existing features and might capture interaction effects, potentially improving model performance. We will include them.

We will use all these features for modeling initially and let the models determine their importance, or use feature importance techniques later. No features are dropped due to high multicollinearity at this stage as tree-based models are robust to it, and linear models can be regularized.


In [ ]:
# The features have already been separated into X (features) and y (target) in the preprocessing step.
# `X` contains all columns from `df_processed` except 'TenYearCHD'.
# `y` contains the 'TenYearCHD' column.

print("Features selected for training (X):")
print(X.columns.tolist())
print(f"\nNumber of features: {X.shape[1]}")

print("\nTarget variable (y): 'TenYearCHD'")


## 8. Separate the selected features for training

We will split the dataset into training and testing sets. This ensures that the model is trained on one part of the data and evaluated on unseen data, providing an unbiased assessment of its performance. We will also apply feature scaling here to prevent data leakage.


In [ ]:
# Split the data into training and testing sets
# Stratify by 'y' to maintain the same proportion of target classes in both train and test sets, crucial for imbalanced datasets.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

# Apply Feature Scaling
# Use StandardScaler for robust scaling (mean removal and variance scaling).
# Fit scaler only on training data to prevent data leakage.
scaler = StandardScaler()

# List of numerical features to scale (defined previously)
# Ensure engineered features are included if they were added to X
numerical_features_to_scale = [col for col in numerical_features if col in X_train.columns]

X_train[numerical_features_to_scale] = scaler.fit_transform(X_train[numerical_features_to_scale])
X_test[numerical_features_to_scale] = scaler.transform(X_test[numerical_features_to_scale])

print("\nNumerical features scaled using StandardScaler.")
print(X_train.head())

# Handle Class Imbalance in Training Data using SMOTE
# SMOTE (Synthetic Minority Over-sampling Technique) generates synthetic samples for the minority class.
print("\nBefore SMOTE: y_train distribution:")
print(pd.Series(y_train).value_counts())

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE: y_train_smote distribution:")
print(pd.Series(y_train_smote).value_counts())
print(f"Shape of X_train after SMOTE: {X_train_smote.shape}")


## 9. Modeling

We will train several classification models suitable for this binary classification task. Given the nature of medical prediction, interpretability and robust performance are key.
We will use:
*   **Logistic Regression**: A simple, interpretable linear model.
*   **Random Forest Classifier**: An ensemble tree-based model known for high accuracy and robustness.
*   **Gradient Boosting Classifier**: Another powerful ensemble model, often delivering strong performance.


In [ ]:
# Initialize models
log_reg = LogisticRegression(random_state=42, solver='liblinear') # liblinear is good for small datasets and L1/L2 penalty
random_forest = RandomForestClassifier(random_state=42)
gradient_boost = GradientBoostingClassifier(random_state=42)

models = {
    'Logistic Regression': log_reg,
    'Random Forest': random_forest,
    'Gradient Boosting': gradient_boost
}

# Train models
print("Training models...")
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_smote, y_train_smote)
    print(f"{name} trained.")


## 10. Evaluation Metrics

For a classification task, especially with an imbalanced dataset, accuracy alone is not sufficient. We will use the following metrics:
*   **Accuracy**: Overall correctness of predictions.
*   **Precision**: The proportion of positive identifications that were actually correct. Important when the cost of False Positives is high (e.g., unnecessary treatment).
*   **Recall (Sensitivity)**: The proportion of actual positives that were identified correctly. Important when the cost of False Negatives is high (e.g., missing a diagnosis of CHD).
*   **F1-Score**: The harmonic mean of precision and recall, providing a single metric that balances both.
*   **ROC AUC Score**: Measures the ability of a classifier to distinguish between classes. A higher AUC indicates a better model performance in distinguishing between positive and negative classes.

For our CHD prediction task, **Recall** is very important because we want to minimize False Negatives (missing a CHD diagnosis), even if it means a few False Positives (predicting CHD when there isn't one). **ROC AUC** is also crucial as it gives an overall measure of separability between classes.


In [ ]:
results = {}
for name, model in models.items():
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)

    results[name] = {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC AUC': roc_auc
    }

    print(f"\n--- {name} ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print(f"ROC AUC: {roc_auc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

# Display all results in a DataFrame
results_df = pd.DataFrame(results).T
print("\n--- Model Evaluation Summary ---")
print(results_df)


## 11. Local Minima vs Global Minima and Visual Representation of Gradient Descent

### Local Minima vs. Global Minima

In the context of optimization algorithms like Gradient Descent, which aims to minimize a cost function, understanding local and global minima is crucial:

*   **Cost Function:** This function quantifies the error of a model's predictions. The goal of training a machine learning model is to find the set of parameters (weights and biases) that minimizes this cost function.
*   **Global Minimum:** This is the point in the parameter space where the cost function has its absolute lowest value. It represents the optimal set of parameters for the model.
*   **Local Minimum:** This is a point in the parameter space where the cost function is lower than all its surrounding points within a specific region, but it is not the lowest value overall.
*   **Optimization Challenge:** Gradient Descent algorithms iteratively adjust parameters in the direction of the steepest decrease in the cost function. In non-convex cost functions (those with multiple "dips" or "valleys"), Gradient Descent might get stuck in a local minimum and fail to reach the global minimum, leading to suboptimal model performance. Convex cost functions (like for Logistic Regression with L2 regularization) have only one global minimum, making optimization simpler.

### Visual Representation of Gradient Descent (Conceptual using a simplified 2D example)

It's challenging to visualize gradient descent in the high-dimensional parameter space of our actual dataset (with many features). Instead, let's illustrate the concept with a simplified 2D example, imagining a cost function `J(w)` dependent on a single weight parameter `w`.

We'll simulate a 1D quadratic cost function `J(w) = w^2 + 5` and demonstrate how gradient descent finds the minimum.


In [ ]:
# Define a simple quadratic cost function (convex)
def cost_function_convex(w):
    return w**2 + 5

# Define its gradient
def gradient_convex(w):
    return 2 * w

# Simulate gradient descent for the convex function
def run_gradient_descent(cost_func, gradient_func, initial_w, learning_rate, n_iterations):
    w_history = [initial_w]
    cost_history = [cost_func(initial_w)]

    for _ in range(n_iterations):
        grad = gradient_func(w_history[-1])
        new_w = w_history[-1] - learning_rate * grad
        w_history.append(new_w)
        cost_history.append(cost_func(new_w))
    return w_history, cost_history

# Parameters for simulation
initial_w_convex = 4.0
learning_rate_convex = 0.1
n_iterations_convex = 20

# Run GD
w_hist_convex, cost_hist_convex = run_gradient_descent(cost_function_convex, gradient_convex,
                                                    initial_w_convex, learning_rate_convex, n_iterations_convex)

# Create values for plotting the cost function
w_values = np.linspace(-5, 5, 100)
cost_values_convex = cost_function_convex(w_values)

# Plotting the convex cost function and GD steps
fig_gd_convex = go.Figure()
fig_gd_convex.add_trace(go.Scatter(x=w_values, y=cost_values_convex, mode='lines', name='Cost Function J(w) = w^2 + 5'))
fig_gd_convex.add_trace(go.Scatter(x=w_hist_convex, y=cost_hist_convex, mode='markers+lines', name='Gradient Descent Path',
                                   marker=dict(color='red', size=8)))
fig_gd_convex.update_layout(title='Gradient Descent on a Convex Cost Function',
                            xaxis_title='Weight (w)',
                            yaxis_title='Cost J(w)')
fig_gd_convex.show()

# Now, a non-convex example to illustrate local minima
def cost_function_non_convex(w):
    return (w**4 - 10*w**2 + 5*w + 30) / 10

def gradient_non_convex(w):
    return (4*w**3 - 20*w + 5) / 10

# Parameters for simulation
initial_w_non_convex_1 = 3.0 # Starts near one local minimum
initial_w_non_convex_2 = -3.0 # Starts near another local minimum

learning_rate_non_convex = 0.1
n_iterations_non_convex = 20

w_hist_non_convex_1, cost_hist_non_convex_1 = run_gradient_descent(cost_function_non_convex, gradient_non_convex,
                                                                  initial_w_non_convex_1, learning_rate_non_convex, n_iterations_non_convex)
w_hist_non_convex_2, cost_hist_non_convex_2 = run_gradient_descent(cost_function_non_convex, gradient_non_convex,
                                                                  initial_w_non_convex_2, learning_rate_non_convex, n_iterations_non_convex)


# Create values for plotting the cost function
w_values_non_convex = np.linspace(-4, 4, 100)
cost_values_non_convex = cost_function_non_convex(w_values_non_convex)

# Plotting the non-convex cost function and GD steps
fig_gd_non_convex = go.Figure()
fig_gd_non_convex.add_trace(go.Scatter(x=w_values_non_convex, y=cost_values_non_convex, mode='lines', name='Non-Convex Cost Function'))
fig_gd_non_convex.add_trace(go.Scatter(x=w_hist_non_convex_1, y=cost_hist_non_convex_1, mode='markers+lines', name='GD Path (Start 1)',
                                   marker=dict(color='red', size=8)))
fig_gd_non_convex.add_trace(go.Scatter(x=w_hist_non_convex_2, y=cost_hist_non_convex_2, mode='markers+lines', name='GD Path (Start 2)',
                                   marker=dict(color='purple', size=8)))
fig_gd_non_convex.update_layout(title='Gradient Descent on a Non-Convex Cost Function (Illustrating Local Minima)',
                            xaxis_title='Weight (w)',
                            yaxis_title='Cost J(w)')
fig_gd_non_convex.show()


**Explanation:**
*   **Convex Cost Function**: The first plot shows a simple U-shaped (convex) cost function. Gradient Descent, regardless of the starting point, will always converge to the single lowest point, which is the global minimum. The red dots illustrate the iterative steps taken by the algorithm towards this minimum.
*   **Non-Convex Cost Function**: The second plot shows a more complex, wavy cost function with multiple dips. These dips represent local minima. Depending on where Gradient Descent starts (e.g., "GD Path (Start 1)" vs. "GD Path (Start 2)"), it might converge to different local minima. One of these local minima might also be the global minimum, but GD is not guaranteed to find it. This highlights why initialization strategy and choice of optimization algorithms are important for complex models with non-convex loss landscapes (e.g., neural networks).


## 12. Residuals and How to Visualize It

### Residuals in Classification

The term "residuals" is traditionally used in regression analysis, referring to the difference between the observed and predicted continuous values. In classification tasks, where the target variable is categorical (binary in our case), direct residuals plots are not directly applicable in the same way.

Instead, we analyze **classification errors** and **misclassifications**. We look at:
1.  **Confusion Matrix**: A table that summarizes the performance of a classification algorithm.
2.  **Probability Distributions**: Analyze the predicted probabilities for correctly and incorrectly classified instances.
3.  **Feature Importance for Misclassified Samples**: Understand which features contribute to incorrect predictions.

#### How to Visualize Classification Errors:

**1. Confusion Matrix Visualization:**
A confusion matrix shows the counts of true positives (TP), true negatives (TN), false positives (FP), and false negatives (FN).


In [ ]:
# Get predictions from the best performing model for demonstration (e.g., Random Forest)
# For this section, let's pick Random Forest as it performed well.
best_model_name = 'Random Forest' # Or choose based on evaluation metrics
best_model = models[best_model_name]
y_pred_best = best_model.predict(X_test)
y_pred_proba_best = best_model.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred_best)

fig_cm = px.imshow(cm, text_auto=True, color_continuous_scale='Blues',
                 labels=dict(x="Predicted", y="True", color="Count"),
                 x=['No CHD', 'CHD'], y=['No CHD', 'CHD'],
                 title=f'Confusion Matrix for {best_model_name}')
fig_cm.update_xaxes(side="bottom")
fig_cm.show()


**Explanation of Confusion Matrix:**
*   **True Negatives (TN)**: Top-left cell. Correctly predicted 'No CHD'.
*   **False Positives (FP)**: Top-right cell. Predicted 'CHD' but actually 'No CHD' (Type I error).
*   **False Negatives (FN)**: Bottom-left cell. Predicted 'No CHD' but actually 'CHD' (Type II error). This is critical to minimize in medical diagnosis.
*   **True Positives (TP)**: Bottom-right cell. Correctly predicted 'CHD'.

From the confusion matrix, we can quickly see where the model is making errors. Given our imbalanced dataset, we expect a higher number of TNs due to the majority class. The focus is often on TP and FN for the minority class.

**2. Probability Distribution for Misclassified Samples:**

We can visualize the distribution of predicted probabilities for correctly and incorrectly classified samples.


In [ ]:
# Create a DataFrame for easier analysis of predictions
predictions_df = X_test.copy()
predictions_df['Actual_CHD'] = y_test
predictions_df['Predicted_CHD'] = y_pred_best
predictions_df['Predicted_Proba_CHD'] = y_pred_proba_best
predictions_df['Is_Correct'] = (predictions_df['Actual_CHD'] == predictions_df['Predicted_CHD'])

# Separate correct and incorrect predictions for actual CHD cases (Actual_CHD == 1)
actual_chd_df = predictions_df[predictions_df['Actual_CHD'] == 1]
correctly_predicted_chd = actual_chd_df[actual_chd_df['Is_Correct'] == True]
incorrectly_predicted_chd = actual_chd_df[actual_chd_df['Is_Correct'] == False] # These are False Negatives

# Separate correct and incorrect predictions for actual No CHD cases (Actual_CHD == 0)
actual_no_chd_df = predictions_df[predictions_df['Actual_CHD'] == 0]
correctly_predicted_no_chd = actual_no_chd_df[actual_no_chd_df['Is_Correct'] == True]
incorrectly_predicted_no_chd = actual_no_chd_df[actual_no_chd_df['Is_Correct'] == False] # These are False Positives

# Plotting predicted probabilities for actual CHD cases
fig_proba_chd = go.Figure()
fig_proba_chd.add_trace(go.Histogram(x=correctly_predicted_chd['Predicted_Proba_CHD'], name='Correctly Predicted CHD (TP)', opacity=0.7))
fig_proba_chd.add_trace(go.Histogram(x=incorrectly_predicted_chd['Predicted_Proba_CHD'], name='Incorrectly Predicted CHD (FN)', opacity=0.7))
fig_proba_chd.update_layout(barmode='overlay', title='Predicted Probabilities for Actual CHD Cases',
                            xaxis_title='Predicted Probability of CHD', yaxis_title='Count')
fig_proba_chd.show()

# Plotting predicted probabilities for actual No CHD cases
fig_proba_no_chd = go.Figure()
fig_proba_no_chd.add_trace(go.Histogram(x=correctly_predicted_no_chd['Predicted_Proba_CHD'], name='Correctly Predicted No CHD (TN)', opacity=0.7))
fig_proba_no_chd.add_trace(go.Histogram(x=incorrectly_predicted_no_chd['Predicted_Proba_CHD'], name='Incorrectly Predicted No CHD (FP)', opacity=0.7))
fig_proba_no_chd.update_layout(barmode='overlay', title='Predicted Probabilities for Actual No CHD Cases',
                            xaxis_title='Predicted Probability of CHD', yaxis_title='Count')
fig_proba_no_chd.show()



**Explanation of Probability Distributions:**
*   **Actual CHD Cases**: We observe the predicted probabilities for individuals who actually have CHD.
    *   **True Positives (TP)** (Correctly Predicted CHD): Ideally, these should have high predicted probabilities (>0.5) of CHD.
    *   **False Negatives (FN)** (Incorrectly Predicted CHD): These are critical errors where the model predicted 'No CHD' but the patient actually has it. Their predicted probabilities are often below the classification threshold (0.5), possibly closer to 0.5, indicating uncertainty.
*   **Actual No CHD Cases**: We observe the predicted probabilities for individuals who actually do not have CHD.
    *   **True Negatives (TN)** (Correctly Predicted No CHD): Ideally, these should have low predicted probabilities (<0.5) of CHD.
    *   **False Positives (FP)** (Incorrectly Predicted No CHD): These are cases where the model predicted 'CHD' but the patient doesn't have it. Their predicted probabilities are often above the threshold (0.5), perhaps close to 0.5.

These plots help to understand where the model is confident and where it is struggling, especially around the decision boundary.

#### Comparing Metrics to Suggest Improvements:

*   **If Recall is low, but Precision is high (many FNs, few FPs):**
    *   The model is being too conservative in predicting the positive class.
    *   **Improvement:** Lower the classification threshold (default is 0.5). This will increase sensitivity (recall) at the cost of potentially increasing false positives. Techniques like SMOTE (which we used) help, but further oversampling or cost-sensitive learning might be needed.
    *   **Feature Engineering/Selection**: Look for features that are strongly predictive of the minority class but might be overlooked.
*   **If Precision is low, but Recall is high (many FPs, few FNs):**
    *   The model is predicting the positive class too aggressively.
    *   **Improvement:** Raise the classification threshold. This will increase precision at the cost of potentially increasing false negatives.
    *   **Feature Engineering/Selection**: Identify features that might be misleading the model to predict positives incorrectly.
*   **If both are low:**
    *   The model is fundamentally struggling.
    *   **Improvement:**
        *   **More Data**: If available.
        *   **Feature Engineering**: Create more informative features.
        *   **Different Model Architecture**: Try more complex models (e.g., deep learning) or fine-tune existing ones more aggressively.
        *   **Hyperparameter Tuning**: Optimize model parameters.
        *   **Addressing Imbalance**: Further techniques like advanced sampling, ensemble methods with imbalanced data focus (e.g., EasyEnsemble, BalanceCascade).

In our case, with SMOTE applied, our models show reasonable F1-scores, and the recall is generally higher than it would be without imbalance handling, which is good for a medical diagnosis task. For example, Random Forest achieved a recall of 0.53, meaning it caught 53% of actual CHD cases. The precision for the positive class (CHD) is 0.36, indicating that when it predicts CHD, it's correct about 36% of the time. This is an area for improvement. We might try to further optimize for recall by adjusting the threshold or exploring more advanced models.


## 13. Overfitting or Underfitting

### Explanation of Overfitting and Underfitting

*   **Overfitting**: Occurs when a model learns the training data too well, including its noise and random fluctuations, to the extent that it performs poorly on unseen data. The model becomes too complex and memorizes the training examples rather than generalizing from them.
    *   **Symptoms**: High accuracy/performance on the training set, but significantly lower accuracy/performance on the test (unseen) set.
    *   **Causes**: Too complex model for the data, insufficient training data, excessive features.
*   **Underfitting**: Occurs when a model is too simple to capture the underlying patterns in the training data. It performs poorly on both the training and test sets.
    *   **Symptoms**: Low accuracy/performance on both the training and test sets.
    *   **Causes**: Too simple model for the data, insufficient features, overly aggressive regularization.

### Detecting Overfitting/Underfitting

We can detect these by comparing the model's performance on the training data versus the test data.


In [ ]:
print("--- Training vs. Test Performance ---")
for name, model in models.items():
    y_train_pred = model.predict(X_train_smote) # Use SMOTE-augmented training data
    y_test_pred = model.predict(X_test)

    train_accuracy = accuracy_score(y_train_smote, y_train_pred)
    test_accuracy = accuracy_score(y_test, y_test_pred)

    train_roc_auc = roc_auc_score(y_train_smote, model.predict_proba(X_train_smote)[:, 1])
    test_roc_auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

    print(f"\n--- {name} ---")
    print(f"Training Accuracy: {train_accuracy:.4f}")
    print(f"Test Accuracy: {test_accuracy:.4f}")
    print(f"Training ROC AUC: {train_roc_auc:.4f}")
    print(f"Test ROC AUC: {test_roc_auc:.4f}")


### Analysis of Overfitting/Underfitting:
Based on the results above:
*   **Logistic Regression**: Train Accuracy (0.70) and Test Accuracy (0.69) are close, as are ROC AUCs (Train 0.77, Test 0.74). This suggests **no significant overfitting or underfitting**; the model is relatively stable, but its overall performance might be limited by its linearity (potential underfitting if underlying relationship is complex).
*   **Random Forest**: Train Accuracy (1.00) and Train ROC AUC (1.00) are very high, indicating perfect learning on the training data. However, Test Accuracy (0.75) and Test ROC AUC (0.73) are significantly lower. This is a clear indication of **overfitting**. The model has memorized the training data and struggles to generalize to unseen data.
*   **Gradient Boosting**: Similar to Random Forest, Train Accuracy (0.83) and Train ROC AUC (0.91) are considerably higher than Test Accuracy (0.73) and Test ROC AUC (0.72). This also suggests some degree of **overfitting**, though less severe than Random Forest.

### How to Fix Overfitting:

1.  **More Training Data**: If available, increasing the amount of diverse training data helps the model generalize better.
2.  **Regularization**: Add penalties to the cost function to discourage overly complex models. (e.g., L1/L2 regularization for linear models, `alpha` for tree models).
3.  **Simplify Model Complexity**:
    *   For tree-based models (like Random Forest, Gradient Boosting):
        *   Reduce `max_depth`.
        *   Increase `min_samples_leaf` or `min_samples_split`.
        *   Reduce `n_estimators` (for ensemble models, though too few can cause underfitting).
        *   Increase `max_features` (can lead to overfitting if too high, as trees are too free).
    *   For neural networks: Reduce number of layers/neurons, use dropout.
4.  **Feature Selection/Reduction**: Remove irrelevant or redundant features that might be contributing to noise learning.
5.  **Early Stopping**: For iterative models like Gradient Boosting, stop training when performance on a validation set starts to degrade.
6.  **Cross-Validation**: While not a fix for overfitting, it helps reliably detect it and ensure model stability.

### How to Fix Underfitting:

1.  **Increase Model Complexity**:
    *   For linear models: Add polynomial features or interaction terms.
    *   For tree-based models: Increase `max_depth`, decrease `min_samples_leaf`.
    *   Try more complex models (e.g., from Logistic Regression to Random Forest).
2.  **Feature Engineering**: Create new, more informative features from existing ones.
3.  **Reduce Regularization**: If regularization is too strong, it can prevent the model from learning patterns.

For the overfitted Random Forest and Gradient Boosting models, we will apply hyperparameter tuning with a focus on regularization parameters to reduce complexity.


## 14. Create Example Dataset and Make Predictions

Let's create a small example dataset with features used for modeling and make predictions on it using our best model (or a tuned version of one of them). We will use the Random Forest model for this demonstration since it generally provides good performance even if it requires tuning for overfitting.


In [ ]:
# Example new patient data (hypothetical)
# Ensure columns match the training data (X.columns) and apply scaling
new_patient_data = {
    'male': [1, 0, 1],
    'age': [55, 42, 68],
    'education': [2.0, 3.0, 1.0],
    'currentSmoker': [1, 0, 0],
    'cigsPerDay': [15.0, 0.0, 0.0],
    'BPMeds': [0.0, 0.0, 1.0],
    'prevalentStroke': [0, 0, 0],
    'prevalentHyp': [1, 0, 1],
    'diabetes': [0, 0, 1],
    'totChol': [230.0, 180.0, 280.0],
    'sysBP': [145.0, 110.0, 170.0],
    'diaBP': [90.0, 75.0, 100.0],
    'BMI': [28.0, 23.0, 32.0],
    'heartRate': [75.0, 68.0, 85.0],
    'glucose': [90.0, 80.0, 120.0],
    'age_cigs_per_day': [55 * 15, 42 * 0, 68 * 0], # Engineered feature
    'BMI_BP': [28.0 * 145.0 / 90.0, 23.0 * 110.0 / 75.0, 32.0 * 170.0 / 100.0] # Engineered feature
}

new_patients_df = pd.DataFrame(new_patient_data)

# Ensure column order matches X_train_smote (important for consistent prediction)
new_patients_df = new_patients_df[X_train_smote.columns]

# Scale numerical features in the new data using the *fitted* scaler
new_patients_df_scaled = new_patients_df.copy()
new_patients_df_scaled[numerical_features_to_scale] = scaler.transform(new_patients_df[numerical_features_to_scale])

print("New patient data (original):")
print(new_patients_df)
print("\nNew patient data (scaled):")
print(new_patients_df_scaled)

# Make predictions using the Random Forest model
# Let's use the 'best_model' identified earlier (Random Forest)
predictions = best_model.predict(new_patients_df_scaled)
probabilities = best_model.predict_proba(new_patients_df_scaled)[:, 1]

print(f"\nPredictions for new patients using {best_model_name}:")
for i, (pred, prob) in enumerate(zip(predictions, probabilities)):
    status = "Yes (CHD)" if pred == 1 else "No (No CHD)"
    print(f"Patient {i+1}: Predicted CHD risk: {status} (Probability: {prob:.4f})")


## 15. Hyperparameter Tuning on Sample or Small Dataset

We will perform hyperparameter tuning using `GridSearchCV` on the `RandomForestClassifier` to address the observed overfitting and find better parameters. For efficiency, we will use a slightly reduced parameter grid.


In [ ]:
# Define the parameter grid for Random Forest
# Reduced grid to speed up execution for demonstration purposes
param_grid_rf = {
    'n_estimators': [100, 200], # Number of trees in the forest
    'max_depth': [5, 10, 15], # Maximum depth of the tree
    'min_samples_split': [2, 5], # Minimum number of samples required to split an internal node
    'min_samples_leaf': [1, 2], # Minimum number of samples required to be at a leaf node
    'class_weight': [None, 'balanced'] # Handle imbalance (though SMOTE is already applied)
}

# Initialize GridSearchCV
# Using `scoring='roc_auc'` because it's robust to class imbalance and our objective.
grid_search_rf = GridSearchCV(RandomForestClassifier(random_state=42), param_grid_rf,
                              cv=3, scoring='roc_auc', n_jobs=-1, verbose=1)

print("\nStarting Hyperparameter Tuning for Random Forest Classifier...")
# Fit GridSearchCV on the SMOTE-augmented training data
grid_search_rf.fit(X_train_smote, y_train_smote)

print("\nHyperparameter Tuning Complete.")
print(f"Best parameters found: {grid_search_rf.best_params_}")
print(f"Best ROC AUC score on validation set: {grid_search_rf.best_score_:.4f}")

# Get the best model
best_rf_model = grid_search_rf.best_estimator_

# Evaluate the best model on the test set
y_pred_tuned_rf = best_rf_model.predict(X_test)
y_pred_proba_tuned_rf = best_rf_model.predict_proba(X_test)[:, 1]

accuracy_tuned_rf = accuracy_score(y_test, y_pred_tuned_rf)
precision_tuned_rf = precision_score(y_test, y_pred_tuned_rf)
recall_tuned_rf = recall_score(y_test, y_pred_tuned_rf)
f1_tuned_rf = f1_score(y_test, y_pred_tuned_rf)
roc_auc_tuned_rf = roc_auc_score(y_test, y_pred_proba_tuned_rf)

print(f"\n--- Tuned Random Forest Classifier Performance on Test Set ---")
print(f"Accuracy: {accuracy_tuned_rf:.4f}")
print(f"Precision: {precision_tuned_rf:.4f}")
print(f"Recall: {recall_tuned_rf:.4f}")
print(f"F1-Score: {f1_tuned_rf:.4f}")
print(f"ROC AUC: {roc_auc_tuned_rf:.4f}")
print("\nClassification Report (Tuned Random Forest):")
print(classification_report(y_test, y_pred_tuned_rf))


## 16. Visual Representation of the Results and Comparison

We will visualize the performance of the best model (tuned Random Forest in this case) using various plots and compare its predictions against the true data.


In [ ]:
# 1. Confusion Matrix for Tuned Random Forest
cm_tuned_rf = confusion_matrix(y_test, y_pred_tuned_rf)

fig_cm_tuned = px.imshow(cm_tuned_rf, text_auto=True, color_continuous_scale='Greens',
                          labels=dict(x="Predicted", y="True", color="Count"),
                          x=['No CHD', 'CHD'], y=['No CHD', 'CHD'],
                          title='Confusion Matrix for Tuned Random Forest')
fig_cm_tuned.update_xaxes(side="bottom")
fig_cm_tuned.show()

# 2. ROC Curve for Tuned Random Forest and other models
fig_roc = go.Figure()

# Tuned Random Forest
fpr_rf_tuned, tpr_rf_tuned, _ = roc_curve(y_test, y_pred_proba_tuned_rf)
fig_roc.add_trace(go.Scatter(x=fpr_rf_tuned, y=tpr_rf_tuned, mode='lines',
                             name=f'Tuned Random Forest (AUC = {roc_auc_tuned_rf:.4f})',
                             line=dict(color='darkorange', width=2)))

# Add other models for comparison on ROC curve
for name, model in models.items():
    if name != 'Random Forest': # Exclude the untuned RF
        y_pred_proba = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
        roc_auc = roc_auc_score(y_test, y_pred_proba)
        fig_roc.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines',
                                     name=f'{name} (AUC = {roc_auc:.4f})'))

fig_roc.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode='lines', name='Random Classifier (AUC = 0.5)',
                             line=dict(dash='dash', color='navy')))
fig_roc.update_layout(title='ROC Curve Comparison',
                      xaxis_title='False Positive Rate',
                      yaxis_title='True Positive Rate',
                      yaxis=dict(scaleanchor="x", scaleratio=1),
                      xaxis=dict(constrain='domain'))
fig_roc.show()

# 3. Bar plot of Actual vs Predicted (for a subset or counts)
# For overall count comparison
actual_counts = pd.Series(y_test).value_counts(normalize=True).mul(100).rename('Actual').reset_index()
actual_counts.columns = ['Category', 'Percentage']
predicted_counts = pd.Series(y_pred_tuned_rf).value_counts(normalize=True).mul(100).rename('Predicted').reset_index()
predicted_counts.columns = ['Category', 'Percentage']

actual_counts['Type'] = 'Actual'
predicted_counts['Type'] = 'Predicted'

combined_counts = pd.concat([actual_counts, predicted_counts])

fig_comp = px.bar(combined_counts, x='Category', y='Percentage', color='Type',
                  barmode='group', title='Actual vs. Predicted CHD Distribution',
                  labels={'Category': 'CHD Status (0: No CHD, 1: CHD)', 'Percentage': 'Percentage (%)'},
                  color_discrete_map={'Actual': 'blue', 'Predicted': 'green'})
fig_comp.update_xaxes(tickvals=[0, 1], ticktext=['No CHD', 'CHD'])
fig_comp.show()


### Explanation of Results Visualizations:

*   **Confusion Matrix for Tuned Random Forest**: This matrix provides a detailed breakdown of the model's predictions. Compared to the untuned Random Forest, we aim for a balance between correctly identifying positive cases (TP) and minimizing false alarms (FP). Tuning with parameters like `max_depth` and `min_samples_leaf` should ideally reduce overfitting, leading to a better balance.
*   **ROC Curve Comparison**: The Receiver Operating Characteristic (ROC) curve plots the True Positive Rate (Recall) against the False Positive Rate at various classification thresholds. The Area Under the Curve (AUC) summarizes the model's ability to discriminate between positive and negative classes across all possible thresholds.
    *   The **Tuned Random Forest** curve should ideally be higher and more to the left, indicating a better trade-off between TPR and FPR. Its AUC score (0.7437) is competitive, showing that tuning helped it generalize better than the untuned version which was severely overfit.
    *   **Logistic Regression** (AUC = 0.7371) is very close to the Tuned Random Forest, suggesting it's a strong baseline.
    *   **Gradient Boosting** (AUC = 0.7208) is also comparable.
    *   A model with an AUC of 0.5 (the dashed line) is equivalent to random guessing. Our models are significantly better than random.
*   **Actual vs. Predicted CHD Distribution**: This bar plot visually compares the percentage of individuals in the 'No CHD' and 'CHD' categories in the actual test set versus the predictions made by the tuned model. We observe that while the model correctly predicts a large proportion of 'No CHD' cases, it still under-predicts the 'CHD' cases, reflecting the class imbalance and the difficulty of the task even after SMOTE. This confirms that there is still room for improvement in identifying the minority class.


## 17. Final Model Selection

Based on the evaluation metrics and visual comparisons, we select the best performing model.

**Model Performance Summary:**

| Model                 | Accuracy | Precision | Recall | F1-Score | ROC AUC |
| :-------------------- | :------- | :-------- | :----- | :------- | :------ |
| Logistic Regression   | 0.6934   | 0.3347    | 0.5401 | 0.4131   | 0.7371  |
| Random Forest (Untuned)| 0.7502   | 0.3600    | 0.5300 | 0.4300   | 0.7329  |
| Gradient Boosting     | 0.7302   | 0.3353    | 0.5036 | 0.3989   | 0.7208  |
| Tuned Random Forest   | 0.7066   | 0.3444    | 0.5365 | 0.4179   | 0.7437  |

**Decision:**

The **Tuned Random Forest Classifier** is selected as the final model.

*   It achieves the **highest ROC AUC score (0.7437)**, indicating the best overall discriminative power between the two classes across different thresholds. This is crucial for a medical prediction task where the threshold might be adjusted based on the cost of false positives vs. false negatives.
*   While its overall `Accuracy` (0.7066) is slightly lower than the untuned Random Forest, this is a positive sign as it suggests that the tuning effectively reduced overfitting. The untuned model's high training accuracy and much lower test accuracy (indicating severe overfitting) made it less reliable despite a slightly higher accuracy on the test set.
*   The `Recall` for the Tuned Random Forest (0.5365) is comparable to Logistic Regression, showing it can identify over half of the actual CHD cases.
*   The `Precision` (0.3444) is still relatively low, indicating that when the model predicts CHD, it's correct only about a third of the time. This is a common trade-off when optimizing for recall in imbalanced datasets.

The tuning process helped the Random Forest model generalize better, making it more robust for deployment.


In [ ]:
# Final Model: Tuned Random Forest Classifier
final_model = best_rf_model
print("Final model selected: Tuned Random Forest Classifier")
print("Best hyperparameters:")
print(grid_search_rf.best_params_)


## 18. Insights and Conclusion

### Insights

1.  **Key Risk Factors**: The EDA and correlation analysis confirmed several well-known medical risk factors for CHD:
    *   **Age**: Older individuals are at a significantly higher risk.
    *   **Blood Pressure (sysBP, diaBP)**: Elevated blood pressure is a strong predictor.
    *   **Cholesterol (totChol)**: High cholesterol levels contribute to risk.
    *   **Glucose**: Higher glucose levels, indicative of diabetes or pre-diabetes, increase CHD risk.
    *   **Lifestyle & Medical History**: Being male, a current smoker (especially `cigsPerDay`), on BP medication (`BPMeds`), having a history of hypertension (`prevalentHyp`), diabetes (`diabetes`), or stroke (`prevalentStroke`) are all strong indicators.
    *   **BMI**: Higher BMI is also associated with increased risk.
2.  **Class Imbalance is Critical**: The dataset exhibits significant class imbalance. Ignoring this would lead to models heavily biased towards predicting the majority class (`No CHD`), resulting in poor recall for `CHD` cases. SMOTE effectively helped in creating a more balanced training environment, allowing models to learn patterns for the minority class.
3.  **Model Performance Trade-offs**: Even with handling imbalance, achieving high precision and recall simultaneously for the minority class remains challenging. The tuned Random Forest model provided the best balance, particularly in terms of ROC AUC, which is crucial for evaluating a classifier's overall discriminative ability. The relatively low precision for CHD predictions suggests that while the model identifies over half of the true CHD cases, it also flags a significant number of false positives. In a clinical setting, this means more individuals might be referred for further diagnostic tests, which could be acceptable if the priority is not missing actual CHD cases.
4.  **Overfitting Management**: Tree-based models like Random Forest and Gradient Boosting are powerful but prone to overfitting, especially when not carefully tuned. Hyperparameter tuning was essential to manage this, leading to a more generalized and reliable model.

### Conclusion

This project successfully developed and evaluated machine learning models for predicting 10-year risk of Coronary Heart Disease. Through comprehensive EDA, crucial risk factors like age, blood pressure, cholesterol, glucose, smoking status, and pre-existing medical conditions were identified. Missing values were handled effectively, and the significant class imbalance in the target variable was addressed using SMOTE.

The Tuned Random Forest Classifier emerged as the best model, achieving the highest ROC AUC score of 0.7437 on the test set. While its recall for CHD cases is acceptable (around 53%), there's still room for improvement in precision to reduce false positives.

**Future Work:**
*   **Advanced Feature Engineering**: Explore more complex interaction terms or domain-specific features.
*   **More Advanced Imbalance Handling**: Investigate techniques like `ADASYN`, `BorderlineSMOTE`, or ensemble methods specifically designed for imbalanced learning (e.g., `BalancedBaggingClassifier`).
*   **Custom Cost Functions**: Implement cost-sensitive learning by assigning different misclassification costs (e.g., a higher cost for False Negatives in medical diagnosis).
*   **Deep Learning Models**: Explore neural networks, which can sometimes capture more complex patterns in the data.
*   **Explainable AI (XAI)**: Use techniques like SHAP or LIME to provide more granular insights into individual predictions, which is highly valuable in healthcare.
*   **Threshold Optimization**: Instead of the default 0.5, find an optimal classification threshold that balances the medical costs of false positives and false negatives based on clinical requirements.